# Reinforcement Learning from Verifiable Rewards with GRPO

After SFT and preference optimization, the model knows how to follow instructions and has been shaped toward human-preferred outputs. GRPO (Group Relative Policy Optimization) is the final reinforcement learning step: the policy is optimized directly against a reward signal — typically a reward model trained on preference data — without any supervised targets. The key insight behind GRPO is that instead of maintaining a separate value network to estimate baselines (as in PPO), we compute advantages [*within a group*]{.underline} of sampled responses for the same prompt. This makes GRPO dramatically cheaper than PPO while retaining most of its stability.

## The REINFORCE Baseline

The policy gradient theorem gives us an unbiased gradient estimator for the expected reward:

$$\nabla_\theta \mathbb{E}_{y \sim \pi_\theta}[r(y)] = \mathbb{E}_{y \sim \pi_\theta}\left[ r(y) \nabla_\theta \log \pi_\theta(y) \right].$$

In practice the reward signal $r(y)$ has high variance — some prompts are inherently harder than others, and raw reward values are not centered. Subtracting a **baseline** $b$ from the reward reduces variance without introducing bias:

$$\nabla_\theta \mathbb{E}_{y \sim \pi_\theta}[r(y)] = \mathbb{E}_{y \sim \pi_\theta}\left[ (r(y) - b) \nabla_\theta \log \pi_\theta(y) \right].$$

Any constant baseline $b$ that does not depend on the sampled $y$ is valid. The standard choice in PPO is a learned value network $V_\phi(x)$ that estimates the expected reward for prompt $x$. GRPO avoids this by computing the baseline from the rewards of the group itself.

## Group Relative Advantages

For each prompt $x$, GRPO samples $G$ responses $\{y_1, \ldots, y_G\}$ from the current policy and scores each with the reward model. The **group relative advantage** for response $y_i$ is:

$$\hat{A}_i = \frac{r_i - \text{mean}(\mathbf{r})}{\text{std}(\mathbf{r}) + \varepsilon}$$

where $\mathbf{r} = (r_1, \ldots, r_G)$. This is z-score normalization within the group: advantages have zero mean and unit standard deviation, and the baseline adapts automatically to each prompt's difficulty. No value network, no separate advantage estimation step.

Implementing `compute_group_advantages`:

In [ ]:
import torch


def compute_group_advantages(rewards: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Normalize rewards within a group to produce zero-mean, unit-std advantages.

    Args:
        rewards: shape (G,) — reward for each response in the group.
        eps: small constant for numerical stability.

    Returns:
        advantages: shape (G,), with mean ≈ 0 and std ≈ 1.
    """
    mean = rewards.mean()
    std = rewards.std() + eps
    return (rewards - mean) / std

## The GRPO Objective

GRPO combines two ingredients: a **clipped surrogate objective** (borrowed from PPO) and a **KL penalty** that keeps the policy close to a frozen reference.

**Clipped surrogate.** Let $\rho_i = \pi_\theta(y_i \mid x) / \pi_{\text{old}}(y_i \mid x)$ be the importance weight comparing the current policy to the policy that generated the samples. The clipped surrogate objective is:

$$L_{\text{clip}}(\theta) = \mathbb{E}_i \left[ \min\!\left(\rho_i \hat{A}_i,\; \text{clip}(\rho_i, 1 - \varepsilon, 1 + \varepsilon)\, \hat{A}_i\right) \right].$$

The clip prevents the policy from updating too aggressively when the importance weight is large.

**KL penalty.** To prevent reward hacking, GRPO also penalizes divergence from a frozen reference policy $\pi_{\text{ref}}$ (the SFT model before RL fine-tuning):

$$D_{\text{KL}}(\pi_\theta \,\|\, \pi_{\text{ref}}) \approx \frac{1}{|y|} \sum_{t=1}^{|y|} \left( \log \pi_\theta(y_t) - \log \pi_{\text{ref}}(y_t) \right).$$

**Full GRPO objective:**

$$\boxed{\mathcal{L}_{\text{GRPO}}(\theta) = -L_{\text{clip}}(\theta) + \beta \cdot D_{\text{KL}}(\pi_\theta \,\|\, \pi_{\text{ref}})}$$

Minimizing $\mathcal{L}_{\text{GRPO}}$ increases the clipped surrogate reward while keeping the policy close to the reference.

The KL divergence is estimated per-token and summed over the response. Note this is the [reverse KL]{.underline} — penalizing the policy for assigning different probabilities than the reference — which encourages the policy to stay close to the reference distribution.

Implementing `kl_divergence_estimate`:

In [ ]:
def kl_divergence_estimate(
    policy_log_probs: torch.Tensor,
    ref_log_probs: torch.Tensor,
) -> torch.Tensor:
    """Estimate per-response KL divergence KL(policy || ref) summed over tokens.

    This is a per-token approximation: sum_t (log pi_theta(y_t) - log pi_ref(y_t)).

    Args:
        policy_log_probs: shape (T,) — log-probs under the current policy.
        ref_log_probs: shape (T,) — log-probs under the frozen reference.

    Returns:
        Scalar tensor — KL estimate summed over the response.
    """
    return (policy_log_probs - ref_log_probs).sum()

## Implementation

GRPO training has four concrete steps per prompt:

1. **Sample** $G$ responses from the current policy.
2. **Score** each response with the reward model.
3. **Compute log-probabilities** under the policy and a frozen reference.
4. **Compute and minimize the GRPO loss.**

We implement each step as a standalone function, then compose them in the training loop.

### Step 1: Sample $G$ responses per prompt

For each training prompt we draw $G$ independent responses from the current policy using ancestral sampling at temperature $T$. All $G$ responses are generated for the same prompt, forming the group used for advantage normalization.

**Configuration.** All GRPO hyperparameters are bundled in a dataclass for easy experiment tracking:

In [ ]:
from dataclasses import dataclass, field


@dataclass
class GRPOConfig:
    # Sampling
    G: int = 8                    # responses per prompt
    max_new_tokens: int = 200     # max generation length
    temperature: float = 0.9      # sampling temperature

    # Objective
    clip_eps: float = 0.2         # PPO clip epsilon
    kl_coef: float = 0.04         # beta — KL penalty coefficient

    # Optimization
    max_steps: int = 2000
    eval_every: int = 200
    batch_size: int = 1           # prompts per gradient step
    lr: float = 1e-5
    lora_rank: int = 8

    # I/O
    output_dir: str = "grpo_checkpoints"

### Step 2: Score the responses

Each of the $G$ sampled responses is scored independently by the reward model. The reward model takes the full sequence (prompt + response) as input and returns a scalar. We collect scores into a tensor for advantage computation.

In [ ]:
def score_responses(
    reward_model,
    tokenizer,
    prompt_ids: torch.Tensor,
    response_ids_list: list[torch.Tensor],
    device: torch.device,
) -> torch.Tensor:
    """Score a list of response tensors using the reward model.

    Args:
        reward_model: model with a scalar head; returns (loss, reward) or just reward.
        tokenizer: tokenizer for padding.
        prompt_ids: shape (prompt_len,) — shared prompt token ids.
        response_ids_list: list of G tensors, each shape (response_len_i,).
        device: compute device.

    Returns:
        rewards: shape (G,) — scalar reward for each response.
    """
    rewards = []
    reward_model.eval()
    with torch.no_grad():
        for response_ids in response_ids_list:
            input_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0)
            input_ids = input_ids.to(device)
            reward = reward_model(input_ids)          # (1,) scalar
            rewards.append(reward.squeeze().cpu())
    return torch.stack(rewards)

### Step 3: Compute log-probabilities

For each response we need the per-token log-probabilities under both the current policy and the frozen reference. We run the full sequence (prompt + response) through the model and collect the log-softmax values at the response positions — masking the prompt positions, following the same convention as SFT.

The reference model is a copy of the SFT checkpoint frozen at the start of GRPO training. Comparing to [NB09](/courses/llm/09-preference-optimization.html) — `get_response_log_probs` is essentially identical to `sequence_log_probs`; the difference is that here we return the per-token tensor rather than summing it, since both the clipped surrogate and KL penalty need per-token access.

In [ ]:
def get_response_log_probs(
    model,
    prompt_ids: torch.Tensor,
    response_ids: torch.Tensor,
    device: torch.device,
) -> torch.Tensor:
    """Compute per-token log-probabilities for response tokens.

    Args:
        model: language model (policy or reference).
        prompt_ids: shape (prompt_len,).
        response_ids: shape (response_len,).
        device: compute device.

    Returns:
        log_probs: shape (response_len,) — log p(y_t | x, y_{<t}) for each token.
    """
    input_ids = torch.cat([prompt_ids, response_ids], dim=0).unsqueeze(0).to(device)
    prompt_len = prompt_ids.size(0)

    with torch.no_grad():
        logits = model(input_ids)                        # (1, T, V)

    log_probs_all = torch.log_softmax(logits[0], dim=-1) # (T, V)

    # Shift: logits at position t predict token at position t+1.
    # Response tokens start at index prompt_len in input_ids.
    response_log_probs = log_probs_all[prompt_len - 1 : -1]  # (response_len, V)
    return response_log_probs.gather(
        dim=-1,
        index=response_ids.unsqueeze(-1).to(device)
    ).squeeze(-1)                                        # (response_len,)

### Step 4: The GRPO loss

With advantages, policy log-probs, old log-probs, and reference log-probs in hand, we assemble the GRPO loss. The importance weight $\rho = \exp(\log \pi_\theta - \log \pi_{\text{old}})$ is computed in log-space for numerical stability. We then apply PPO-style clipping and add the KL penalty.

In [ ]:
def grpo_loss(
    policy_log_probs: torch.Tensor,
    old_log_probs: torch.Tensor,
    ref_log_probs: torch.Tensor,
    advantages: torch.Tensor,
    cfg: GRPOConfig,
) -> tuple[torch.Tensor, dict]:
    """Compute the GRPO loss for a single (prompt, response) pair.

    Args:
        policy_log_probs: shape (T,) — log-probs under current policy (with grad).
        old_log_probs: shape (T,) — log-probs under sampling policy (no grad).
        ref_log_probs: shape (T,) — log-probs under frozen reference (no grad).
        advantages: scalar — precomputed group-relative advantage for this response.
        cfg: GRPOConfig hyperparameters.

    Returns:
        loss: scalar tensor (differentiable).
        metrics: dict with per-step diagnostics.
    """
    # Importance weight (clipped surrogate)
    log_ratio = policy_log_probs.sum() - old_log_probs.sum()  # <1>
    ratio = log_ratio.exp()
    clipped_ratio = ratio.clamp(1 - cfg.clip_eps, 1 + cfg.clip_eps)

    surrogate = torch.min(ratio * advantages, clipped_ratio * advantages)  # <2>

    # KL penalty
    kl = kl_divergence_estimate(policy_log_probs, ref_log_probs)  # <3>

    loss = -surrogate + cfg.kl_coef * kl

    metrics = {
        "ratio": ratio.item(),
        "clip_fraction": (ratio.detach().abs() > 1 + cfg.clip_eps).float().item(),
        "kl": kl.item(),
        "advantage": advantages.item(),
    }
    return loss, metrics

1. Log-ratio $\log \rho = \sum_t \log \pi_\theta(y_t) - \sum_t \log \pi_{\text{old}}(y_t)$; exponentiated to get importance weight.
2. Clipped surrogate: take the more conservative (smaller) of the unclipped and clipped objectives.
3. Per-token KL summed over the response.

## The GRPO Training Loop

The GRPO loop is structurally different from SFT. The outer loop iterates over prompts; the inner loop samples $G$ responses, scores them, computes advantages, and accumulates one gradient update per prompt. The policy and reference share the same base weights — only the LoRA adapters are trained. The reference is frozen throughout.

**Setup:** LoRA is injected into the SFT model checkpoint (see [NB08](/courses/llm/08-sft-lora.html)). The reference is a separate copy of the same SFT checkpoint with all parameters frozen. Both models are loaded before training begins.

In [ ]:
import copy
import os
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset


def grpo_train(
    sft_model_path: str,
    reward_model_path: str,
    prompt_dataset_path: str,
    cfg: GRPOConfig,
) -> dict:
    """Full GRPO training loop.

    Loads SFT model + LoRA adapters as the policy, a frozen copy as the reference,
    and a trained reward model. Trains the policy by optimizing the GRPO objective
    against the reward model signal.

    Args:
        sft_model_path: path to SFT model checkpoint.
        reward_model_path: path to trained reward model checkpoint.
        prompt_dataset_path: path to dataset of prompts (no labels needed).
        cfg: GRPOConfig.

    Returns:
        history: dict with keys 'reward', 'kl', 'entropy', 'clip_frac',
                 'reward_std', 'step' — one entry per eval step.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # --- Load models ---
    policy = torch.load(sft_model_path, map_location=device)
    inject_lora(policy, rank=cfg.lora_rank)                 # <1>
    policy.train()

    ref_model = copy.deepcopy(policy)
    for p in ref_model.parameters():                       # <2>
        p.requires_grad_(False)
    ref_model.eval()

    reward_model = torch.load(reward_model_path, map_location=device)
    reward_model.eval()

    optimizer = AdamW(
        [p for p in policy.parameters() if p.requires_grad],
        lr=cfg.lr,
    )

    # --- Load prompts ---
    tokenizer = load_tokenizer()                            # <3>
    dataset = PromptDataset(prompt_dataset_path, tokenizer)
    loader = DataLoader(dataset, batch_size=cfg.batch_size, shuffle=True)

    history = {k: [] for k in ["step", "reward", "kl", "entropy", "clip_frac", "reward_std"]}
    step = 0

    while step < cfg.max_steps:
        for batch in loader:
            if step >= cfg.max_steps:
                break

            prompt_ids = batch["input_ids"][0]              # <4>

            # Step 1: Sample G responses
            policy.eval()
            response_ids_list = [
                policy.generate(
                    prompt_ids.unsqueeze(0).to(device),
                    max_new_tokens=cfg.max_new_tokens,
                    temperature=cfg.temperature,
                    do_sample=True,
                )[0, prompt_ids.size(0):].cpu()
                for _ in range(cfg.G)
            ]
            policy.train()

            # Step 2: Score
            rewards = score_responses(
                reward_model, tokenizer, prompt_ids, response_ids_list, device
            )

            # Compute advantages
            advantages = compute_group_advantages(rewards)  # <5>

            # Step 3 + 4: Log-probs and loss
            old_log_probs_list = []
            with torch.no_grad():
                for resp_ids in response_ids_list:
                    old_lp = get_response_log_probs(policy, prompt_ids, resp_ids, device)
                    old_log_probs_list.append(old_lp)

            total_loss = torch.tensor(0.0, device=device)
            step_metrics = []

            for i, (resp_ids, adv) in enumerate(zip(response_ids_list, advantages)):
                policy_lp = get_response_log_probs.__wrapped__(policy, prompt_ids, resp_ids, device)  # <6>
                ref_lp = get_response_log_probs(ref_model, prompt_ids, resp_ids, device)

                # Re-run policy with grad
                input_ids = torch.cat([prompt_ids, resp_ids]).unsqueeze(0).to(device)
                logits = policy(input_ids)
                log_probs_all = torch.log_softmax(logits[0], dim=-1)
                plen = prompt_ids.size(0)
                policy_lp_grad = log_probs_all[plen - 1:-1].gather(
                    -1, resp_ids.unsqueeze(-1).to(device)
                ).squeeze(-1)

                loss_i, m = grpo_loss(
                    policy_lp_grad,
                    old_log_probs_list[i].to(device),
                    ref_lp.to(device),
                    adv.to(device),
                    cfg,
                )
                total_loss = total_loss + loss_i / cfg.G  # <7>
                step_metrics.append(m)

            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            step += 1

            # Logging
            if step % cfg.eval_every == 0:
                mean_reward = rewards.mean().item()
                mean_kl = sum(m["kl"] for m in step_metrics) / cfg.G
                mean_clip = sum(m["clip_fraction"] for m in step_metrics) / cfg.G
                reward_std = rewards.std().item()
                ent = response_entropy(policy, prompt_ids, device=device)

                history["step"].append(step)
                history["reward"].append(mean_reward)
                history["kl"].append(mean_kl)
                history["clip_frac"].append(mean_clip)
                history["reward_std"].append(reward_std)
                history["entropy"].append(ent)

                print(
                    f"step {step:4d} | reward {mean_reward:.3f} "
                    f"| kl {mean_kl:.4f} | entropy {ent:.3f}"
                )

    # Save checkpoint
    os.makedirs(cfg.output_dir, exist_ok=True)
    torch.save(policy.state_dict(), os.path.join(cfg.output_dir, "policy_final.pt"))
    return history

1. `inject_lora` adds LoRA adapters to all attention projection layers; only adapter parameters have `requires_grad=True`.
2. The reference model is a deep copy of the policy before LoRA — frozen throughout training. All parameters have `requires_grad=False`.
3. `load_tokenizer()` and `PromptDataset` are thin wrappers matching the tokenizer used during SFT.
4. `batch_size=1` — one prompt per gradient step. Each prompt generates $G$ responses for advantage normalization.
5. Group relative advantages: zero mean, unit std across the $G$ rewards.
6. Re-running with gradients for the policy forward pass — `get_response_log_probs` uses `torch.no_grad()` internally for speed; we inline the forward pass here to get gradients.
7. Average the loss over the $G$ responses in the group.

## Monitoring GRPO Training

GRPO training is harder to monitor than SFT because there is no held-out loss to track directly. Instead, we watch four signals:

- **Reward** — should increase over training. Flat or decreasing reward suggests the policy has converged or the reward model signal is too noisy.
- **KL divergence** — should stay bounded. Exploding KL means the policy has drifted far from the reference; the KL penalty coefficient `kl_coef` should be increased.
- **Response entropy** — should remain reasonably high. Entropy collapse (near zero) means the policy has degenerated to outputting a single response regardless of prompt.
- **Reward standard deviation** within each group — should decrease as the policy becomes more consistent. If it stays high, the policy is still sampling diverse outputs and has room to improve.

In [ ]:
def response_entropy(
    model,
    prompt_ids: torch.Tensor,
    n_samples: int = 16,
    max_new_tokens: int = 50,
    device: torch.device = None,
) -> float:
    """Estimate generation entropy by sampling n_samples responses and measuring
    token-level entropy from the model's output distribution.

    Args:
        model: current policy.
        prompt_ids: shape (prompt_len,) — single prompt.
        n_samples: number of samples for entropy estimation.
        max_new_tokens: length of each sampled response.
        device: compute device.

    Returns:
        Mean per-token entropy (nats).
    """
    if device is None:
        device = next(model.parameters()).device

    model.eval()
    total_entropy = 0.0
    total_tokens = 0

    with torch.no_grad():
        for _ in range(n_samples):
            input_ids = prompt_ids.unsqueeze(0).to(device)
            for _ in range(max_new_tokens):
                logits = model(input_ids)[0, -1, :]       # (V,)
                probs = torch.softmax(logits, dim=-1)
                entropy = -(probs * (probs + 1e-12).log()).sum()
                total_entropy += entropy.item()
                total_tokens += 1
                next_token = torch.multinomial(probs, 1)
                input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=1)

    model.train()
    return total_entropy / max(total_tokens, 1)

The training history is visualized in a 2×3 panel showing all four signals plus clip fraction and reward std:

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt


def plot_grpo_training(history: dict) -> None:
    """Plot GRPO training diagnostics in a 2x3 panel.

    Args:
        history: dict with keys 'step', 'reward', 'kl', 'entropy',
                 'clip_frac', 'reward_std'.
    """
    steps = history["step"]
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    fig.suptitle("GRPO Training Diagnostics", fontsize=13)

    panels = [
        ("reward",     "Mean Reward",            "reward"),
        ("kl",         "KL Divergence",           "kl"),
        ("entropy",    "Response Entropy (nats)", "entropy"),
        ("clip_frac",  "Clip Fraction",           "clip_frac"),
        ("reward_std", "Reward Std (within group)","reward_std"),
    ]

    for ax, (key, title, _) in zip(axes.flat, panels):
        ax.plot(steps, history[key], linewidth=1.5)
        ax.set_title(title)
        ax.set_xlabel("Step")
        ax.grid(True, alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

    # Hide unused sixth panel
    axes.flat[-1].set_visible(False)

    plt.tight_layout()
    plt.show()

## Reward Hacking

Reward hacking is the central failure mode in RL fine-tuning: the policy learns to [exploit the reward model]{.mark} rather than producing genuinely better outputs. Since the reward model is imperfect, the policy can find degenerate inputs — repetitive phrases, grammatical nonsense, gibberish — that achieve high reward scores while degrading generation quality.

Three diagnostic symptoms:

1. **Reward climbs while output quality degrades** — reward keeps rising, but human evaluators find the outputs worse. The policy has found a reward model blind spot.
2. **Entropy collapse** — generation entropy approaches zero. The policy outputs the same response (or near-same) regardless of prompt, having converged to a reward-hacking mode.
3. **KL divergence explodes then reward drops** — the policy drifts far from the reference, overshooting. The KL penalty then pushes back, causing reward to drop sharply.

In [ ]:
def detect_reward_hacking(
    history: dict,
    policy,
    reward_model,
    tokenizer,
    prompts: list[str],
    device: torch.device,
    kl_threshold: float = 0.5,
    entropy_threshold: float = 1.0,
    reward_window: int = 5,
) -> list[str]:
    """Detect reward hacking symptoms from training history.

    Checks three conditions:
    1. KL divergence above threshold.
    2. Response entropy below threshold.
    3. Reward has been decreasing for the last `reward_window` steps.

    Args:
        history: training history dict.
        policy: current policy model.
        reward_model: reward model.
        tokenizer: tokenizer.
        prompts: small set of evaluation prompts for live entropy estimation.
        device: compute device.
        kl_threshold: KL above this value triggers a warning.
        entropy_threshold: entropy below this value triggers a warning.
        reward_window: number of recent steps to check for reward decrease.

    Returns:
        warnings: list of warning strings (empty if no symptoms detected).
    """
    warnings = []

    # Symptom 1: KL explosion
    if history["kl"] and history["kl"][-1] > kl_threshold:
        warnings.append(
            f"[WARNING] KL divergence = {history['kl'][-1]:.4f} > threshold {kl_threshold}. "
            f"Consider increasing kl_coef."
        )

    # Symptom 2: entropy collapse
    recent_entropy = []
    for prompt_text in prompts[:4]:
        prompt_ids = tokenizer.encode(prompt_text, return_tensors="pt")[0]
        ent = response_entropy(policy, prompt_ids, device=device)
        recent_entropy.append(ent)
    mean_ent = sum(recent_entropy) / max(len(recent_entropy), 1)
    if mean_ent < entropy_threshold:
        warnings.append(
            f"[WARNING] Mean generation entropy = {mean_ent:.3f} < threshold {entropy_threshold}. "
            f"Possible entropy collapse."
        )

    # Symptom 3: reward decreasing
    if len(history["reward"]) >= reward_window:
        recent = history["reward"][-reward_window:]
        if recent[-1] < recent[0]:
            warnings.append(
                f"[WARNING] Reward decreased over last {reward_window} eval steps "
                f"({recent[0]:.3f} → {recent[-1]:.3f}). Possible reward hacking overshoot."
            )

    for w in warnings:
        print(w)

    return warnings

### Early Stopping

GRPO training should stop automatically when reward hacking symptoms appear. `GRPOEarlyStopper` checks all three conditions after every eval step and halts training if any are triggered.

In [ ]:
class GRPOEarlyStopper:
    """Early stopping for GRPO based on reward hacking symptoms.

    Stops training if:
    - KL divergence exceeds ``kl_threshold``.
    - Response entropy falls below ``entropy_threshold``.
    - Mean reward has not improved for ``patience`` consecutive eval steps.

    Args:
        kl_threshold: max allowable KL divergence.
        entropy_threshold: min allowable generation entropy.
        patience: number of non-improving eval steps before stopping.
    """

    def __init__(
        self,
        kl_threshold: float = 0.5,
        entropy_threshold: float = 1.0,
        patience: int = 5,
    ) -> None:
        self.kl_threshold = kl_threshold
        self.entropy_threshold = entropy_threshold
        self.patience = patience
        self._best_reward = float("-inf")
        self._non_improving = 0

    def step(self, reward: float, kl: float, entropy: float) -> bool:
        """Check stopping conditions after one eval step.

        Args:
            reward: mean reward at this eval step.
            kl: mean KL divergence at this eval step.
            entropy: mean response entropy at this eval step.

        Returns:
            True if training should stop, False otherwise.
        """
        if kl > self.kl_threshold:
            print(f"[EarlyStopper] KL {kl:.4f} > {self.kl_threshold}. Stopping.")
            return True
        if entropy < self.entropy_threshold:
            print(f"[EarlyStopper] Entropy {entropy:.3f} < {self.entropy_threshold}. Stopping.")
            return True

        if reward > self._best_reward:
            self._best_reward = reward
            self._non_improving = 0
        else:
            self._non_improving += 1
            if self._non_improving >= self.patience:
                print(
                    f"[EarlyStopper] Reward did not improve for {self.patience} steps. Stopping."
                )
                return True

        return False

## GRPO vs PPO

GRPO and PPO optimize the same underlying objective — the KL-penalized expected reward from the RLHF paper — but differ in how they estimate the baseline for variance reduction.

**PPO** maintains a learned value network $V_\phi(x)$ that estimates the expected return from each state. This provides low-variance advantage estimates but requires a second neural network with its own optimizer, approximately doubling the memory and compute cost.

**GRPO** replaces the value network with within-group reward normalization. No extra parameters are trained. The trade-off is higher variance: with small $G$, the group mean is a noisy baseline. But for language model tasks — where prompts are short and responses are long — $G = 8$–$16$ is often sufficient to get stable training.

| | PPO | GRPO |
|---|---|---|
| Baseline | Learned value network $V_\phi$ | Within-group mean reward |
| Extra parameters | Yes — value head | No |
| Variance | Low | Higher (decreases with $G$) |
| Memory | ~2× policy | ~1× policy |
| Implementation complexity | High | Moderate |
| Used in | InstructGPT, many LLMs | DeepSeekMath, DeepSeek-R1 |

: GRPO vs PPO comparison. {tbl-colwidths="[25, 35, 35]"}

GRPO was introduced in the DeepSeekMath paper [@shao2024] specifically to reduce the cost of RL fine-tuning for math reasoning tasks. DeepSeek-R1 [@deepseek2025] subsequently scaled GRPO to the full pretraining regime, demonstrating that group-relative advantages are sufficient for training long-form reasoning models.

## Summary

| Concept | Key detail |
|---|---|
| REINFORCE | Policy gradient $\nabla_\theta J = \mathbb{E}[R \nabla \log \pi_\theta]$; high variance without baseline. |
| Group relative advantage | z-score within $G$ samples: $(R_i - \mu_G) / \sigma_G$. [Eliminates the value network entirely]{.mark}. |
| Clipped surrogate | $\min(\rho A,\; \text{clip}(\rho, 1\pm\varepsilon) A)$ — limits each update to a trust region. |
| KL penalty | $\beta\, D_{\text{KL}}(\pi_\theta \| \pi_{\text{ref}})$ keeps the policy close to the SFT checkpoint. |
| Verifiable rewards | Binary correctness signal (e.g., regex match) — no learned reward model needed. |
| Reward hacking | Policy exploits reward loopholes. Monitor output length and diversity as early indicators. |
| GRPO vs PPO | GRPO trades lower variance for $G\times$ more generation; PPO trades a value network for lower sample cost. |

: {tbl-colwidths="[30,70]"}

## Exercises

1. **Group size ablation.** Run `grpo_train` with $G \in \{2, 4, 8, 16\}$ on a small prompt dataset. Plot final reward vs $G$. At what group size does the advantage estimate stabilize?

2. **KL coefficient sweep.** Train with `kl_coef` $\in \{0.0, 0.01, 0.04, 0.1\}$. Compare final reward, KL divergence, and response diversity. What happens with `kl_coef=0.0`?

3. **Reward hacking simulation.** Train a reward model that gives high scores to responses containing a specific token pattern (e.g., responses ending in "!"). Observe `GRPOEarlyStopper` triggering as the policy learns to append "!" to every response.

4. **Entropy-based early stopping.** Implement a version of `response_entropy` that uses the full vocabulary distribution at each step (not just sampled tokens). Compare the entropy estimate to the sampling-based version.

5. **GRPO without clipping.** Remove the `clip_eps` clipping and set `clip_eps=∞`. Compare training stability to clipped GRPO. When does unclipped GRPO diverge?

6. **Reference model update.** Some implementations periodically update the reference model to the current policy ("reference model refresh"). Implement this and compare to a fixed reference. Does reward hacking happen more or less quickly?

■